<a href="https://colab.research.google.com/github/AndreaFreemanfse/Flatiron/blob/main/Prompt_Engineering_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt

Context:

    Scenario: The product team needs to segment customers by purchasing behavior for a new feature rollout.

    Database Schema:

    user_activity
     user_id
    last_login_date
    feature_usage_count
    account_type
    transactions
     transaction_id
    user_id
    transaction_date
    amount
    platform
    user_preferences
     user_id
    communication_preference
    interface_theme
    notification_settings
    Task: Create a SQL query to identify:

    Active users (logged in last 30 days)
    Filter by high-value customers (top 20% by spending)
    User preference trends for the identified customers

Intructions:

    Write a complete, runnable Python script using `sqlite3` that creates test tables and inserts sample data for user activity, transactions, and user preferences. Then write SQL queries to identify users who logged in within the last 30 days and belong to the top 20% of customers by total spending. Include their account details and preferences, then summarize preference trends among those high-value customers. Use CTEs and window functions where appropriate, handle date filtering with SQLite date functions, and print the results. Include all imports, comments, and database connection cleanup so I can copy and run the script as is. Also briefly explain how the percentile calculation, date filtering, and query optimization work.

# Response

In [2]:
import sqlite3

# ============================================
# 1. CONNECT TO SQLITE DATABASE
# ============================================

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# ============================================
# 2. CREATE TABLES
# ============================================

cursor.executescript("""
CREATE TABLE user_activity (
    user_id INTEGER PRIMARY KEY,
    last_login_date TEXT NOT NULL,
    feature_usage_count INTEGER,
    account_type TEXT
);

CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY,
    user_id INTEGER REFERENCES user_activity(user_id),
    transaction_date TEXT,
    amount REAL,
    platform TEXT
);

CREATE TABLE user_preferences (
    user_id INTEGER PRIMARY KEY REFERENCES user_activity(user_id),
    communication_preference TEXT,
    interface_theme TEXT,
    notification_settings TEXT
);
""")

# ============================================
# 3. INSERT TEST DATA
# ============================================

cursor.executemany("""
INSERT INTO user_activity VALUES (?, date('now', ?), ?, ?)
""", [
    (1, '-2 days', 25, 'Premium'),
    (2, '-5 days', 18, 'Standard'),
    (3, '-10 days', 40, 'Premium'),
    (4, '-15 days', 12, 'Standard'),
    (5, '-1 day', 50, 'Premium'),
    (6, '-45 days', 8, 'Standard'),
    (7, '-7 days', 35, 'Premium'),
    (8, '-20 days', 10, 'Standard'),
    (9, '-3 days', 45, 'Premium'),
    (10, '-60 days', 5, 'Standard')
])

cursor.executemany("""
INSERT INTO transactions VALUES (?, ?, date('now', ?), ?, ?)
""", [
    (101, 1, '-10 days', 500.00, 'Web'),
    (102, 1, '-5 days', 300.00, 'Mobile'),
    (103, 2, '-8 days', 100.00, 'Web'),
    (104, 3, '-4 days', 900.00, 'Mobile'),
    (105, 3, '-2 days', 600.00, 'Web'),
    (106, 4, '-6 days', 150.00, 'Web'),
    (107, 5, '-1 day', 1200.00, 'Mobile'),
    (108, 6, '-40 days', 2000.00, 'Web'),
    (109, 7, '-3 days', 700.00, 'Mobile'),
    (110, 8, '-12 days', 200.00, 'Web'),
    (111, 9, '-2 days', 400.00, 'Mobile'),
    (112, 10, '-50 days', 3000.00, 'Web')
])

cursor.executemany("""
INSERT INTO user_preferences VALUES (?, ?, ?, ?)
""", [
    (1, 'Email', 'Dark', 'Enabled'),
    (2, 'SMS', 'Light', 'Enabled'),
    (3, 'Email', 'Dark', 'Enabled'),
    (4, 'Push', 'Light', 'Disabled'),
    (5, 'Email', 'Dark', 'Enabled'),
    (6, 'SMS', 'Dark', 'Disabled'),
    (7, 'Push', 'Dark', 'Enabled'),
    (8, 'Email', 'Light', 'Enabled'),
    (9, 'Email', 'Light', 'Disabled'),
    (10, 'SMS', 'Dark', 'Enabled')
])

# ============================================
# 4. IDENTIFY ACTIVE, HIGH-VALUE CUSTOMERS
# ============================================

customer_query = """
WITH active_users AS (
    SELECT *
    FROM user_activity
    WHERE last_login_date >= date('now', '-30 days')
      AND last_login_date <= date('now')
),
customer_spending AS (
    SELECT
        t.user_id,
        SUM(t.amount) AS total_spending
    FROM transactions t
    JOIN active_users a ON a.user_id = t.user_id
    GROUP BY t.user_id
),
ranked_customers AS (
    SELECT
        a.user_id,
        a.account_type,
        a.last_login_date,
        a.feature_usage_count,
        cs.total_spending,
        ROW_NUMBER() OVER (
            ORDER BY cs.total_spending DESC, a.user_id
        ) AS spending_rank,
        COUNT(*) OVER () AS total_customers
    FROM active_users a
    JOIN customer_spending cs ON cs.user_id = a.user_id
),
high_value_customers AS (
    SELECT *
    FROM ranked_customers
    WHERE spending_rank <= CAST(
        CEIL(total_customers * 0.20) AS INTEGER
    )
)
SELECT
    h.user_id,
    h.account_type,
    h.last_login_date,
    h.feature_usage_count,
    h.total_spending,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM high_value_customers h
LEFT JOIN user_preferences p ON p.user_id = h.user_id
ORDER BY h.total_spending DESC;
"""

print("ACTIVE HIGH-VALUE CUSTOMERS")
print("-" * 75)

cursor.execute(customer_query)

for row in cursor.fetchall():
    print(row)

# ============================================
# 5. SUMMARIZE PREFERENCE TRENDS
# ============================================

preferences_query = """
WITH active_users AS (
    SELECT user_id
    FROM user_activity
    WHERE last_login_date >= date('now', '-30 days')
      AND last_login_date <= date('now')
),
customer_spending AS (
    SELECT
        t.user_id,
        SUM(t.amount) AS total_spending
    FROM transactions t
    JOIN active_users a ON a.user_id = t.user_id
    GROUP BY t.user_id
),
ranked_customers AS (
    SELECT
        user_id,
        total_spending,
        ROW_NUMBER() OVER (
            ORDER BY total_spending DESC, user_id
        ) AS spending_rank,
        COUNT(*) OVER () AS total_customers
    FROM customer_spending
),
high_value_customers AS (
    SELECT user_id
    FROM ranked_customers
    WHERE spending_rank <= CAST(
        CEIL(total_customers * 0.20) AS INTEGER
    )
)
SELECT
    p.communication_preference,
    p.interface_theme,
    p.notification_settings,
    COUNT(*) AS customer_count
FROM high_value_customers h
JOIN user_preferences p ON p.user_id = h.user_id
GROUP BY
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
ORDER BY customer_count DESC;
"""

print("\nPREFERENCE TRENDS")
print("-" * 75)

cursor.execute(preferences_query)

for row in cursor.fetchall():
    print(row)

# ============================================
# 6. CLOSE DATABASE CONNECTION
# ============================================

conn.close()
print("\nDatabase connection closed.")

ACTIVE HIGH-VALUE CUSTOMERS
---------------------------------------------------------------------------
(3, 'Premium', '2026-10-13', 40, 1500.0, 'Email', 'Dark', 'Enabled')
(5, 'Premium', '2026-10-22', 50, 1200.0, 'Email', 'Dark', 'Enabled')

PREFERENCE TRENDS
---------------------------------------------------------------------------
('Email', 'Dark', 'Enabled', 2)

Database connection closed.
